# 🚀 Wandb (Weights & Biases) 튜토리얼

**Weights & Biases (Wandb)**는 머신러닝 실험을 추적하고 관리하는 플랫폼입니다.
오늘 실습에서는 학습하는 네트워크/모델의 학습 경향을 Wandb를 이용하여 추적함으로써 개선된 metric를 얻기 위한 튜닝을 해볼 예정입니다.

### 실습 시작 전
1. Wandb 무료 계정 생성
2. [https://wandb.ai/authorize](https://wandb.ai/authorize) API 키 복사


In [ ]:
import os
import wandb

print("Welcome to the Wandb Tutorial!")
print("Please get your API key from: https://wandb.ai/authorize")
print()

try:
    import getpass
    api_key = getpass.getpass("Enter your Wandb API key: ")
except Exception:
    print("Note: Your API key will be visible when typing.")
    api_key = input("Enter your Wandb API key: ")

# Set the API key as an environment variable
os.environ["WANDB_API_KEY"] = api_key

# Login to wandb
try:
    wandb.login(key=api_key, relogin=True)
    print("✅ Successfully logged in to Wandb!")
    print(f"✅ You are logged in as: {wandb.api.default_entity}")
except Exception as e:
    print(f"❌ Login failed: {e}")
    print("Please check your API key and try again.")

# Wandb 함수 이해하기

이 튜토리얼에서는 세 가지 핵심 Wandb 함수를 살펴보겠습니다. 각 함수가 무엇을 하는지 알아봅시다:

## 🚀 `wandb.init()` - 새로운 실행 초기화

이 함수는 새로운 실험 추적 세션을 시작합니다.

**주요 매개변수:**
- `project`: 프로젝트 이름 (관련 실험들을 그룹화)
- `config`: 추적할 하이퍼파라미터 딕셔너리
- `name`: 이 실행에 대한 선택적 사용자 정의 이름
- `tags`: 실행을 정리하기 위한 태그 리스트
- `notes`: 실험에 대한 설명

## 📊 `wandb.log()` - 메트릭과 데이터 로깅

이 함수는 각 단계에서 메트릭, 이미지 또는 기타 데이터를 기록합니다.

**주요 매개변수:**
- 로깅할 메트릭의 딕셔너리 (예: `{"loss": 0.5, "accuracy": 0.9}`)
- `step`: 선택적 단계 번호 (제공되지 않으면 자동 증가)
- `commit`: 로그를 즉시 저장할지 여부 (기본값: True)

**로깅할 수 있는 것들:**
- 스칼라 값 (loss, accuracy 등)
- 이미지, 플롯, 히스토그램
- 오디오, 비디오, 테이블
- 사용자 정의 차트

## ✅ `wandb.finish()` - 실행 종료

이 함수는 현재 실행을 적절히 종료하고 최종 데이터를 서버에 업로드합니다.

In [ ]:
# Example: Simple Machine Learning Training Loop with Wandb

import os, time, random, wandb

# Store wandb files under results/ (creates results/wandb)
os.makedirs("results", exist_ok=True)

# Initialize a new wandb run
wandb.init(
    project="intro-wandb",           # Project name (groups related experiments)
    dir="results",                   # Local directory for wandb files (-> results/wandb)
    config={                         # Hyperparameters to track
        "lr": 1e-3,                 # Learning rate
        "batch_size": 64            # Batch size
    }
)

# Simulate a training loop
print("Starting training simulation...")
for step in range(100):
    # Simulate decreasing loss with some noise
    loss = 1.0/(step+1) + random.random()*0.03
    
    # Log metrics at each step
    wandb.log({
        "loss": loss,               # Training loss
        "step": step               # Training step
    })
    
    # Small delay to simulate training time
    time.sleep(0.05)
    
    # Print progress every 20 steps
    if step % 20 == 0:
        print(f"Step {step}: Loss = {loss:.4f}")

print("Training completed!")

# Properly close the wandb run
wandb.finish()

# 🖼️ 이미지 & 3D 데이터 로깅

wandb는 스칼라(loss, accuracy)뿐 아니라 **이미지, 그래프, 3D 포인트 클라우드** 같은 다양한 미디어도 로깅할 수 있습니다.
NeRF/3DGS 같은 3D 작업에서는 렌더링된 뷰나 복원된 포인트 클라우드를 학습 과정에서 직접 추적할 수 있어 매우 유용합니다.

**주요 미디어 타입:**
- `wandb.Image`: numpy 배열 `(H, W, 3)`, PIL 이미지, matplotlib figure 등을 로깅
- `wandb.Object3D`: 3D 포인트 클라우드를 인터랙티브 뷰어로 로깅
  - `(N, 3)`: xyz 좌표만
  - `(N, 4)`: xyz + 카테고리(label)
  - `(N, 6)`: xyz + rgb (색상은 0~255 범위)

업로드 후에는 Wandb 대시보드에서 이미지를 확인하고, 3D 포인트 클라우드는 마우스로 회전·확대하며 볼 수 있습니다.

In [ ]:
# Example: Logging Images and 3D Point Clouds to Wandb (per-step)

import os
import numpy as np
import matplotlib.pyplot as plt
import wandb

os.makedirs("results", exist_ok=True)

# Start a new run for media logging
wandb.init(
    project="intro-wandb",
    dir="results",
    name="media-logging-demo",
)

# Simulate a training loop that logs media at every step
print("Starting media logging simulation...")
xx, yy = np.meshgrid(np.linspace(-1, 1, 128), np.linspace(-1, 1, 128))

for step in range(20):
    # --- 1) Log an image that evolves over steps (e.g., a rendered view) ---
    phase = step * 0.3
    img = np.stack([
        0.5 + 0.5 * np.sin(4 * xx + phase),
        0.5 + 0.5 * np.cos(4 * yy + phase),
        0.5 + 0.5 * np.sin(4 * (xx + yy) + phase),
    ], axis=-1)

    # --- 2) Log a matplotlib figure that evolves over steps ---
    t = np.linspace(0, 10, 100)
    fig, ax = plt.subplots()
    ax.plot(t, np.sin(t + phase))
    ax.set_title(f"matplotlib figure (step {step})")

    # --- 3) Log a 3D point cloud that rotates/grows over steps ---
    n = 2000
    np_phi = np.random.uniform(0, np.pi, n)
    np_theta = np.random.uniform(0, 2 * np.pi, n) + phase
    x = np.sin(np_phi) * np.cos(np_theta)
    y = np.sin(np_phi) * np.sin(np_theta)
    z = np.cos(np_phi)
    xyz = np.stack([x, y, z], axis=-1)
    rgb = (xyz * 0.5 + 0.5) * 255            # color by position, range 0-255
    points = np.concatenate([xyz, rgb], axis=-1)  # (N, 6): xyz + rgb

    # Log all media for this step together
    wandb.log({
        "sample_image": wandb.Image(img, caption=f"Synthetic RGB image (step {step})"),
        "sample_plot": wandb.Image(fig),
        "point_cloud": wandb.Object3D(points),
    }, step=step)
    plt.close(fig)

    if step % 5 == 0:
        print(f"Logged media at step {step}")

wandb.finish()
print("✅ Logged per-step images, plots, and 3D point clouds to Wandb!")